In [318]:
import pandas as pd

required_files = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
]

datasets = {
    "customers": customers,
    "orders": orders,
    "products": products,
    "order_items": order_items,
}

customers = pd.read_csv("../data/raw/customers.csv")
products = pd.read_csv("../data/raw/products.csv")
orders = pd.read_csv("../data/raw/orders.csv")
order_items = pd.read_csv("../data/raw/order_items.csv")


In [319]:
for name, df in datasets.items():
    print(name, df.shape, df.columns.tolist())

customers (150, 6) ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']
orders (301, 5) ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']
products (100, 4) ['product_id', 'product_name', 'category', 'price']
order_items (766, 6) ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']


In [320]:
customer_basic = customers[[
    "customer_id",
    "gender",
    "age",
    "city",
]]

In [321]:
customers_over_30 = customers[
    customers["age"] >= 30
]

city_customers = customers[
    customers["city"].isin(["서울", "부산"])
]

In [322]:
customers["city"].value_counts()

city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

In [323]:
orders["order_status"].value_counts()

order_status
completed    184
cancelled     65
refunded      52
Name: count, dtype: int64

In [324]:
products.sort_values(
    "price",
    ascending=False,
).head(10)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


In [325]:
order_items = order_items.copy()
order_items["line_total"] = (
    order_items["quantity"]
    * order_items["unit_price"]
)

In [326]:
orders["order_status"].value_counts()

order_status
completed    184
cancelled     65
refunded      52
Name: count, dtype: int64

In [327]:
customers["city"].value_counts()

city
성남    21
광주    17
부산    16
대구    15
서울    15
울산    14
인천    14
대전    14
수원    13
고양    11
Name: count, dtype: int64

In [328]:
products.sort_values(
    "price",
    ascending=False,
).head(10)

,product_id,product_name,category,price
98,99,뷰티 상품 099,뷰티,200000
69,70,패션 상품 070,패션,198000
57,58,식품 상품 058,식품,197000
42,43,뷰티 상품 043,뷰티,197000
23,24,스포츠 상품 024,스포츠,196000
8,9,스포츠 상품 009,스포츠,193000
36,37,뷰티 상품 037,뷰티,193000
71,72,뷰티 상품 072,뷰티,189000
7,8,스포츠 상품 008,스포츠,189000
52,53,생활용품 상품 053,생활용품,188000


In [329]:
#step5. 정렬과 파생 컬럼 만들기
order_items = order_items.copy()
order_items["line_total"] = (
    order_items["quantity"]
    * order_items["unit_price"]
)

print(order_items)

     order_item_id  order_id  product_id  quantity  unit_price  line_total
0                1         1         100         3      102000      306000
1                2         1          87         5       25000      125000
2                3         1           7         3      142000      426000
3                4         1           9         3      193000      579000
4                5         2          72         4      189000      756000
..             ...       ...         ...       ...         ...         ...
761            762       299           8         2      189000      378000
762            763       299          12         4      175000      700000
763            764       300          59         5       32000      160000
764            789       320          67         5       32000      160000
765            763       299         108         4      175000      700000

[766 rows x 6 columns]


In [330]:
#step6. orders와 병합하고 검증하기

print(
    "orders.order_id 중복 수:",
    orders["order_id"].duplicated().sum(),
)

orders.order_id 중복 수: 0


In [331]:
order_sales = order_items.merge(
    orders[[
        "order_id",
        "customer_id",
        "order_date",
        "order_status",
    ]],
    on="order_id",
    how="left",
    validate="many_to_one",
    indicator=True,
)

In [332]:
print("병합 전 행 수:", len(order_items))
print("병합 후 행 수:", len(order_sales))
print(order_sales["_merge"].value_counts())

병합 전 행 수: 766
병합 후 행 수: 766
_merge
both          765
left_only       1
right_only      0
Name: count, dtype: int64


### 병합 되지 않은 행 발견
1. 병합 되지 않은 행만 찾는다.

In [333]:
unmatched_order_items = order_sales.loc[
    order_sales["_merge"].eq("left_only")
]

display(
    unmatched_order_items[
        ["order_item_id", "order_id", "product_id", "quantity", "unit_price", "_merge"]
    ]
)

print(unmatched_order_items)

,order_item_id,order_id,product_id,quantity,unit_price,_merge
764,789,320,67,5,32000,left_only


     order_item_id  order_id  product_id  quantity  unit_price  line_total  \
764            789       320          67         5       32000      160000   

     customer_id order_date order_status     _merge  
764          NaN        NaN          NaN  left_only  


In [334]:
# 1. order_id 320이 orders 테이블에 있는지 찾는다.
# 2. 없으면 주문 항목에만 존재하는 고아 데이터로 판단한다.
#
# 설명:
# left merge 뒤의 NaN은 원인이라기보다,
# 오른쪽 테이블에서 연결할 주문 정보를 찾지 못했다는 결과다.

orders.loc[orders["order_id"].eq(320)]

# orders 테이블에 order_id 320 이 있는지 확인하는 코드로 확인.

,order_id,customer_id,order_date,payment_method,order_status


In [335]:
# orders에 없는 주문 항목 확인
unmatched_order_items = order_sales.loc[
    order_sales["_merge"].eq("left_only")
]

# 정상적으로 주문 정보와 연결된 항목만 분석에 사용
valid_order_sales = order_sales.loc[
    order_sales["_merge"].eq("both")
].copy()

In [336]:
# 1. 분석용 데이터에는 left_only가 없어야 함
print(valid_order_sales["_merge"].value_counts())

_merge
both          765
left_only       0
right_only      0
Name: count, dtype: int64


In [337]:
# 2. 주문 정보 열에 NaN이 없어야 함
print(
    valid_order_sales[
        ["customer_id", "order_date", "order_status"]
    ].isna().sum()
)

customer_id     0
order_date      0
order_status    0
dtype: int64


In [338]:
# 3. 제외된 행이 정확히 1건인지 확인
print("제외된 주문 항목 수:", len(unmatched_order_items))
print(unmatched_order_items[["order_id", "line_total"]])

제외된 주문 항목 수: 1
     order_id  line_total
764       320      160000


## STEP 7. 날짜 변환 후 completed 주문만 선택하기

In [339]:
order_sales["order_date"] = pd.to_datetime(
    order_sales["order_date"],
    errors="coerce",
)

print(
    "날짜 변환 실패:",
    order_sales["order_date"].isna().sum(),
)

날짜 변환 실패: 1


In [340]:
completed_order_sales = order_sales[
    order_sales["order_status"] == "completed"
].copy()

In [341]:
print(completed_order_sales)

     order_item_id  order_id  product_id  quantity  unit_price  line_total  \
0                1         1         100         3      102000      306000   
1                2         1          87         5       25000      125000   
2                3         1           7         3      142000      426000   
3                4         1           9         3      193000      579000   
12              13         6          83         3       24000       72000   
..             ...       ...         ...       ...         ...         ...   
758            759       297          20         5       80000      400000   
759            760       297          42         4       28000      112000   
761            762       299           8         2      189000      378000   
762            763       299          12         4      175000      700000   
765            763       299         108         4      175000      700000   

     customer_id order_date order_status _merge  
0          12

In [342]:
# 월 컬럼을 만든다
completed_order_sales["order_month"] = (
    completed_order_sales["order_date"]
    .dt.to_period("M")
    .astype(str)
)

In [343]:
completed_order_sales[["order_date", "order_month"]].head()

,order_date,order_month
0,2026-06-03,2026-06
1,2026-06-03,2026-06
2,2026-06-03,2026-06
3,2026-06-03,2026-06
12,2026-04-17,2026-04


In [344]:
completed_order_sales["order_status"].unique()

<ArrowStringArray>
['completed']
Length: 1, dtype: str

## STEP 8. products와 병합하고 다시 검증하기

In [345]:
completed_sales_items = completed_order_sales.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    indicator="product_merge",
)

In [346]:
# 주문 정보 병합 결과
completed_sales_items["_merge"].value_counts()

_merge
both          475
left_only       0
right_only      0
Name: count, dtype: int64

In [347]:
print(completed_sales_items.columns.tolist())

['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total', 'customer_id', 'order_date', 'order_status', '_merge', 'order_month', 'product_name', 'category', 'price', 'product_merge']


In [348]:
# 상품 정보 병합 결과
completed_sales_items["product_merge"].value_counts()

product_merge
both          474
left_only       1
right_only      0
Name: count, dtype: int64

In [349]:
print("병합 전:", len(completed_order_sales))
print("병합 후:", len(completed_sales_items))
completed_sales_items[
    ["_merge","product_merge"]
    ].value_counts()

병합 전: 475
병합 후: 475


_merge      product_merge
both        both             474
            left_only          1
            right_only         0
left_only   both               0
            left_only          0
            right_only         0
right_only  both               0
            left_only          0
            right_only         0
Name: count, dtype: int64

In [350]:
product_unmatched = completed_sales_items.loc[
    completed_sales_items["product_merge"].eq("left_only"),
    ["order_id", "product_id", "quantity", "unit_price", "line_total"],
]

display(product_unmatched)

,order_id,product_id,quantity,unit_price,line_total
474,299,108,4,175000,700000


In [351]:
# 상품 정보가 없는 완료 주문 항목 확인
missing_products = completed_sales_items.loc[
    completed_sales_items["product_merge"].eq("left_only")
].copy()

# 상품 정보가 없는 경우에도 매출은 유지하고,
# 분석용 데이터에서만 상품명·카테고리를 표시한다.
completed_sales_items["product_name"] = (
    completed_sales_items["product_name"].fillna("상품 정보 없음")
)

completed_sales_items["category"] = (
    completed_sales_items["category"].fillna("미분류")
)

In [352]:
category_revenue = (
    completed_sales_items.groupby("category", as_index=False)
    .agg(revenue=("line_total", "sum"))
    .sort_values("revenue", ascending=False)
)

In [353]:
# 2. 미분류 처리가 되었는지 확인
display(
    completed_sales_items.loc[
        completed_sales_items["category"].eq("미분류"),
        ["order_id", "product_id", "product_name", "category", "line_total"],
    ]
)

,order_id,product_id,product_name,category,line_total
474,299,108,상품 정보 없음,미분류,700000


In [354]:
# 3. 카테고리별 매출에 미분류가 포함됐는지 확인
display(
    category_revenue.loc[
        category_revenue["category"].eq("미분류")
    ]
)

,category,revenue
1,미분류,700000


## STEP 9. 카테고리별·상품별 요약표 만들기

In [355]:
category_sales = (
    completed_sales_items
    .groupby("category", as_index=False)
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)

category_sales

,category,total_quantity,total_sales
4,스포츠,295,31743000
6,전자기기,259,26400000
3,생활용품,272,23915000
2,뷰티,223,23383000
5,식품,133,16573000
0,도서,149,16389000
7,패션,111,10587000
1,미분류,4,700000


In [356]:
product_sales = (
    completed_sales_items
    .groupby(
        ["product_id", "product_name", "category"],
        as_index=False,
    )
    .agg(
        total_quantity=("quantity", "sum"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)

product_sales

,product_id,product_name,category,total_quantity,total_sales
39,41,스포츠 상품 041,스포츠,35,5705000
11,12,식품 상품 012,식품,25,4375000
8,9,스포츠 상품 009,스포츠,20,3860000
70,72,뷰티 상품 072,뷰티,20,3780000
69,71,전자기기 상품 071,전자기기,23,3703000
...,...,...,...,...,...
95,98,스포츠 상품 098,스포츠,13,130000
92,95,전자기기 상품 095,전자기기,6,120000
33,35,스포츠 상품 035,스포츠,2,118000
5,6,전자기기 상품 006,전자기기,18,90000


In [357]:
print(category_sales["total_sales"].sum())
print(completed_sales_items["line_total"].sum())

149690000
149690000


## STEP 10. 월별·고객별 요약표 만들기

월별 요약:

In [358]:
monthly_summary = (
    completed_order_sales
    .groupby("order_month", as_index=False)
    .agg(
        total_sales=("line_total", "sum"),
        order_count=("order_id", "nunique"),
    )
    .sort_values("order_month")
)

고객별 요약:

In [359]:
customer_sales = (
    completed_order_sales
    .groupby("customer_id", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("total_sales", ascending=False)
)


필요한 고객 속성만 연결:

`customer_sales`에 `.merge()`로 `customers`의 `customer_id`, `city` 열을 합친다.  
연결 기준(`on`)은 `customer_id`이고, 왼쪽 기준(`how="left"`)은 `customer_sales`다.  
`validate="one_to_one"`은 양쪽 표에 같은 고객 번호가 각각 한 번씩만 있어야 한다고 확인하는 검사다.  

`validate`는 pandas `merge()`에 있는 매개변수 다.

즉, 고객별 매출표에 고객 도시 정보를 붙인다

In [360]:
customer_sales = customer_sales.merge(
    customers[["customer_id", "city"]],
    on="customer_id",
    how="left",
    validate="one_to_one",
)


In [361]:
monthly_sales = (
    completed_order_sales
    .groupby("order_month", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_sales=("line_total", "sum"),
    )
    .sort_values("order_month")
)

display(monthly_sales)

,order_month,order_count,total_sales
0,2025-08,9,6582000
1,2025-09,19,16818000
2,2025-10,15,12895000
3,2025-11,25,23611000
4,2025-12,11,8621000
5,2026-01,14,10935000
6,2026-02,21,17154000
7,2026-03,17,9151000
8,2026-04,17,15536000
9,2026-05,20,15402000


## STEP 11. 결과 CSV 저장하고 다시 읽기

In [362]:
from pathlib import Path
import sys

# 수도코드:
# 1. 현재 노트북의 실행 위치를 확인한다.
# 2. 현재 위치와 그 상위 폴더들을 하나씩 검사한다.
# 3. course_utils 폴더가 발견된 위치를 프로젝트 루트로 정한다.
# 4. 프로젝트 루트를 Python 검색 경로에 추가한다.
#
# 설명:
# course_utils는 notebooks 폴더 안이 아니라 프로젝트 루트에 있다.
# 따라서 현재 노트북 위치에서 부모 폴더 방향으로 올라가며 course_utils를 찾는다.

# [1] 현재 노트북이 실행 중인 폴더를 Path 객체로 가져온다.
current = Path.cwd().resolve()

# [2] current는 현재 폴더, current.parents는 상위 폴더들의 목록이다.
# 예: notebooks → 00_llm-data-analysis-course → ai-data-analysis → C:\
for path in (current, *current.parents):

    # [3] 현재 검사 중인 path 안에 course_utils 폴더가 있는지 확인한다.
    if (path / "course_utils").is_dir():

        # [4] course_utils를 가진 폴더를 프로젝트 루트로 저장하고 반복을 끝낸다.
        project_root = path
        break

# 위 반복문이 끝날 때까지 course_utils를 못 찾은 경우 실행된다.
else:
    raise FileNotFoundError("course_utils 폴더를 찾지 못했습니다.")

# [5] 프로젝트 루트가 Python 검색 경로에 없다면 맨 앞에 추가한다.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("현재 실행 위치:", current)
print("프로젝트 루트:", project_root)

현재 실행 위치: C:\dev\ai-data-analysis\00_llm-data-analysis-course\notebooks
프로젝트 루트: C:\dev\ai-data-analysis\00_llm-data-analysis-course


In [363]:
# course_utils/paths.py에 만든 공통 경로 함수를 가져온다.
from course_utils.paths import get_report_dir


# 수도코드:
# 1. 프로젝트의 reports 폴더 경로를 가져온다.
# 2. reports 안에서 Chapter 04 결과 폴더를 지정한다.
# 3. 그 폴더 안의 CSV 파일을 이름순으로 찾는다.
# 4. 각 파일의 이름, 존재 여부, 파일 크기를 출력한다.
#
# 설명:
# get_report_dir()는 프로젝트의 reports 폴더를 반환한다.
# / "chapter04"를 붙이면 Chapter 04 분석 결과를 저장한 폴더를 가리킨다.


# [1] 프로젝트 루트 아래의 reports 폴더 경로를 가져온다.
# 예: .../00_llm-data-analysis-course/reports
report_dir = get_report_dir()


# [2] reports 폴더 안의 chapter04 결과 폴더 경로를 만든다.
# 예: .../00_llm-data-analysis-course/reports/chapter04
chapter04_report_dir = report_dir / "chapter04"


# [3] chapter04 폴더 안에서 확장자가 .csv인 파일을 모두 찾는다.
# sorted()는 파일 이름을 가나다/알파벳 순서로 정렬한다.
for path in sorted(chapter04_report_dir.glob("*.csv")):

    # [4] 찾은 CSV 파일마다 정보를 출력한다.
    print(
        path.name,         # 파일 이름만 출력한다. 예: category_revenue.csv
        path.exists(),     # 해당 파일이 실제로 존재하면 True를 출력한다.
        path.stat().st_size,  # 파일 크기를 바이트(byte) 단위로 출력한다.
    )

# [2-1] 결과 DataFrame과 저장 파일 이름을 연결한다.
reports_to_save = {
    "ch04_category_sales.csv": category_sales,
    "ch04_product_sales.csv": product_sales,
    "ch04_monthly_sales.csv": monthly_sales,
    "ch04_customer_sales.csv": customer_sales,
}

# [3-1] 각 DataFrame을 reports 폴더에 CSV로 저장한다.
for filename, dataframe in reports_to_save.items():
    output_path = report_dir / filename

    dataframe.to_csv(
        output_path,
        index=False,        # DataFrame의 행 번호는 저장하지 않는다.
        encoding="utf-8-sig",
    )

    print("저장 완료:", output_path)


저장 완료: C:\dev\ai-data-analysis\00_llm-data-analysis-course\reports\ch04_category_sales.csv
저장 완료: C:\dev\ai-data-analysis\00_llm-data-analysis-course\reports\ch04_product_sales.csv
저장 완료: C:\dev\ai-data-analysis\00_llm-data-analysis-course\reports\ch04_monthly_sales.csv
저장 완료: C:\dev\ai-data-analysis\00_llm-data-analysis-course\reports\ch04_customer_sales.csv


In [367]:
check_category = pd.read_csv(
    report_dir / "ch04_category_sales.csv"
)

print(check_category.shape)
print(check_category.columns.tolist())

(8, 3)
['category', 'total_quantity', 'total_sales']


.columns.tolist 사용 이유

```python
# Index 객체 그대로 출력
print(check_category.columns)
# Index(['category', 'total_quantity', 'total_sales'], dtype='object')

# 리스트로 간단히 출력
print(check_category.columns.tolist())
# ['category', 'total_quantity', 'total_sales']
```

### [Chapter 04 Evidence]

1. Notebook
- notebooks/ch04_pandas_basic.ipynb 실행 완료: 예

2. 병합 검증
- orders.order_id 중복: 0건
- orders merge 전 행 수: 766
- orders merge 후 행 수: 766
- orders merge 미매칭: left_only 1건
  - order_id=320은 order_items에는 있지만 orders에는 없음
- products merge 전 행 수: 475
- products merge 후 행 수: 475
- products merge 미매칭: left_only 1건
  - order_id=299의 product_id=108은 products에 없음

3. 계산 범위
- 사용 주문 상태: completed
- completed 주문 수: 184건
- line_total 계산식: quantity × unit_price
- 날짜 변환 실패: 0건
  - orders 원본 기준 모든 order_date가 날짜로 변환됨
  - 기존 병합 후 검사에서 나온 1건은 order_id=320 미매칭으로 생긴 NaN이며 날짜 형식 오류가 아님

4. 교차 검증
- 완료 주문 상세 line_total 합계: 149,690,000
- category_sales 합계: 149,690,000
- monthly_summary 합계: 149,690,000
- customer_sales 합계: 149,690,000
- 합계 일치 여부: PASS

5. 저장 파일
- ch04_category_sales.csv: 존재
- ch04_product_sales.csv: 존재
- ch04_monthly_sales.csv: 존재
- ch04_customer_sales.csv: 존재

6. LLM 검증
- 병합 후 날짜 변환 실패를 확인하면 order_id=320 미매칭으로 생긴 NaN도 날짜 오류로 잘못 계산될 수 있었다.
- orders 원본에서 날짜 변환을 먼저 검사해야 실제 날짜 형식 오류와 병합 미매칭을 구분할 수 있다.
- product_id=108 미매칭 상품은 매출을 제거하지 않고, 상품명은 "상품 정보 없음", 카테고리는 "미분류"로 처리했다.
- total_sales는 완료 주문 항목 금액의 합계이며, 환불·할인·세금·배송비를 반영한 회계상 순매출이라고 단정하지 않는다.

7. 남은 질문
- order_id=320과 product_id=108이 원본 마스터 데이터에 누락된 근본 원인을 확인하려면 원본 주문·상품 생성 과정을 추가로 점검해야 한다.

In [368]:
# 수도코드:
# 1. 완료 주문 상세 매출과 각 집계표의 매출 합계를 계산한다.
# 2. 네 합계가 같은지 확인한다.
#
# 설명:
# 같은 완료 주문 매출을 다른 기준으로 집계했으므로,
# 총매출은 모두 같아야 한다.

detail_total = completed_sales_items["line_total"].sum()
category_total = category_sales["total_sales"].sum()
monthly_total = monthly_sales["total_sales"].sum()
customer_total = customer_sales["total_sales"].sum()

print("완료 주문 상세 합계:", detail_total)
print("카테고리 합계:", category_total)
print("월별 합계:", monthly_total)
print("고객별 합계:", customer_total)

assert detail_total == category_total == monthly_total == customer_total
print("합계 일치 여부: PASS")

완료 주문 상세 합계: 149690000
카테고리 합계: 149690000
월별 합계: 149690000
고객별 합계: 149690000
합계 일치 여부: PASS
